# Multimodal Fraud Detection - Complete Pipeline

This notebook demonstrates the complete end-to-end pipeline for multimodal fraud detection with Google Drive integration.

## Pipeline Flow:
```
Online Payment Fraud Dataset (Google Drive)
          ↓
    Tabular Data
          ↓
FraudDataPreprocessor
          ↓
          ├─────────────────────┐
          ↓                     ↓
QR Code Images (Google Drive)   |
          ↓                     |
     Image Data                 |
          ↓                     |
  QRCodePreprocessor            |
          ↓                     |
          └─────────────────────┤
                                ↓
         MultimodalDataPreprocessor
                                ↓
         Processed Data Storage (Google Drive)
                                ↓
          ├─────────────────────┐
          ↓                     ↓
TabularTransformerEncoder  VisionTransformerEncoder
          ↓                     ↓
          └─────────┬───────────┘
                    ↓
         Cross-Modal Fusion
                    ↓
           Fraud Detection
```

## Datasets Used:
1. **Online Payments Fraud Detection Dataset**: Transaction data with fraud labels
2. **Benign and Malicious QR Codes Dataset**: QR code images for visual analysis

All data is stored and loaded from **Google Drive** for persistence.

## 1. Setup and Google Drive Authentication

Mount Google Drive to access and store datasets and processed data.

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Define Google Drive paths for datasets and processed data
DRIVE_BASE = '/content/drive/MyDrive/FraudDetection'
FRAUD_DATASET_PATH = f'{DRIVE_BASE}/datasets/online_payment_fraud'
QR_DATASET_PATH = f'{DRIVE_BASE}/datasets/qr_codes'
PROCESSED_DATA_PATH = f'{DRIVE_BASE}/processed_data'
MODEL_SAVE_PATH = f'{DRIVE_BASE}/models'

# Create directories if they don't exist
os.makedirs(FRAUD_DATASET_PATH, exist_ok=True)
os.makedirs(QR_DATASET_PATH, exist_ok=True)
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print("✓ Google Drive mounted successfully")
print(f"✓ Dataset paths created:")
print(f"  - Fraud Dataset: {FRAUD_DATASET_PATH}")
print(f"  - QR Dataset: {QR_DATASET_PATH}")
print(f"  - Processed Data: {PROCESSED_DATA_PATH}")
print(f"  - Models: {MODEL_SAVE_PATH}")

## 2. Install Dependencies and Clone Repository

In [ ]:
# Install required packages
!pip install -q tensorflow numpy pandas matplotlib scikit-learn pillow

# Clone the repository (if not already cloned)
import os
if not os.path.exists('/content/test'):
    !git clone https://github.com/go2nishantnig/test.git /content/test
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

# Add to Python path
import sys
sys.path.insert(0, '/content/test')

print("✓ All dependencies installed")

## 3. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import pickle
import json
from pathlib import Path

# Import preprocessors
from src.utils.tabular_preprocessor import FraudDataPreprocessor
from src.utils.image_preprocessor import QRCodePreprocessor
from src.utils.multimodal_preprocessor import MultimodalDataPreprocessor

# Import model components
from src.models.transformer import MultimodalFraudDetectionTransformer
from src.models.blocks import TransformerBlock
from src.models.embeddings import PatchEmbedding
from config.config import MODEL_CONFIG, TRAINING_CONFIG

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("✓ All libraries imported successfully")

## 📦 Kaggle Dataset Setup (Optional)

This notebook supports loading data from the real Kaggle datasets:

### 1. Online Payments Fraud Detection Dataset
- **Download**: [Kaggle Dataset](https://www.kaggle.com/datasets/rupakroy/online-payments-fraud-detection-dataset)
- **Place CSV at**: `{FRAUD_DATASET_PATH}/PS_20174392719_1491204439457_log.csv`
- **Format**: CSV file with transaction data
- **Columns**: step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig, nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud

### 2. Benign and Malicious QR Codes Dataset
- **Download**: [Kaggle Dataset](https://www.kaggle.com/datasets/samahsadiq/benign-and-malicious-qr-codes)
- **Expected structure**:
  ```
  {QR_DATASET_PATH}/
    benign/
      *.png
    malicious/
      *.png
  ```
- **Format**: PNG image files in two folders

### Fallback Mode
If the Kaggle datasets are not found, the notebook will automatically generate **synthetic data** for demonstration purposes. This allows you to run the entire pipeline without downloading the datasets.

**Note**: To use the real datasets, download them from Kaggle and upload to the appropriate folders in your Google Drive at the paths shown above.

## 4. Load Online Payment Fraud Dataset from Google Drive

Load or generate the Online Payment Fraud Detection dataset.

In [ ]:
# Load Online Payment Fraud Dataset from Google Drive or Kaggle## This cell supports three modes:# 1. Load from real Kaggle CSV file (if available)# 2. Load from previously saved data (if exists)# 3. Generate synthetic data (fallback)fraud_csv_path = f'{FRAUD_DATASET_PATH}/fraud_transactions.csv'# Option 1: Try to load from Kaggle dataset (if you've downloaded it)# Download from: https://www.kaggle.com/datasets/rupakroy/online-payments-fraud-detection-datasetkaggle_csv_path = f'{FRAUD_DATASET_PATH}/PS_20174392719_1491204439457_log.csv'  # Typical Kaggle filenamefraud_preprocessor_temp = FraudDataPreprocessor()if os.path.exists(kaggle_csv_path):    print("Loading fraud dataset from Kaggle CSV...")    fraud_data = fraud_preprocessor_temp.load_from_csv(kaggle_csv_path)        # Standardize column name for consistency    if 'isFraud' in fraud_data.columns and 'is_fraud' not in fraud_data.columns:        fraud_data['is_fraud'] = fraud_data['isFraud']        # Save processed version for faster loading next time    fraud_data.to_csv(fraud_csv_path, index=False)    print(f"✓ Loaded {len(fraud_data)} transactions from Kaggle dataset")    print(f"✓ Saved to {fraud_csv_path} for faster loading next time")    # Option 2: Load from previously saved dataelif os.path.exists(fraud_csv_path):    print("Loading existing fraud dataset from Google Drive...")    fraud_data = pd.read_csv(fraud_csv_path)    print(f"✓ Loaded {len(fraud_data)} transactions from Drive")    # Option 3: Generate synthetic data as fallbackelse:    print("No existing dataset found. Generating synthetic fraud data...")    print("Tip: Download the Kaggle dataset and place it at:")    print(f"  {kaggle_csv_path}")        # Generate synthetic fraud transaction data    fraud_data = fraud_preprocessor_temp.generate_synthetic_data(        n_samples=5000,        fraud_ratio=0.1  # 10% fraud transactions    )        # Save to Google Drive for future use    fraud_data.to_csv(fraud_csv_path, index=False)    print(f"✓ Generated and saved {len(fraud_data)} transactions to Drive")# Ensure consistent column namingif 'isFraud' in fraud_data.columns and 'is_fraud' not in fraud_data.columns:    fraud_data['is_fraud'] = fraud_data['isFraud']# Display dataset infoprint("\n" + "="*60)print("ONLINE PAYMENT FRAUD DATASET (Tabular Data)")print("="*60)print(f"Dataset shape: {fraud_data.shape}")print(f"Fraud ratio: {fraud_data['is_fraud'].mean():.2%}")print(f"\nColumns: {list(fraud_data.columns)}")print(f"\nFirst few rows:")display(fraud_data.head())# Visualize fraud distributionfig, axes = plt.subplots(1, 2, figsize=(12, 4))# Fraud countfraud_data['is_fraud'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])axes[0].set_title('Fraud vs Normal Transactions')axes[0].set_xlabel('Transaction Type')axes[0].set_ylabel('Count')axes[0].set_xticklabels(['Normal', 'Fraud'], rotation=0)# Amount distributionaxes[1].hist(fraud_data[fraud_data['is_fraud']==0]['amount'],              alpha=0.5, label='Normal', bins=30, color='green')axes[1].hist(fraud_data[fraud_data['is_fraud']==1]['amount'],              alpha=0.5, label='Fraud', bins=30, color='red')axes[1].set_title('Transaction Amount Distribution')axes[1].set_xlabel('Amount')axes[1].set_ylabel('Frequency')axes[1].legend()plt.tight_layout()plt.show()

## 5. Load QR Code Images Dataset from Google Drive

Load or generate QR code images for fraud detection.

In [ ]:
# Load QR Code Images Dataset from Google Drive or Kaggle## This cell supports three modes:# 1. Load from real Kaggle image folders (if available)# 2. Load from previously saved numpy arrays (if exists)# 3. Generate synthetic data (fallback)qr_images_path = f'{QR_DATASET_PATH}/qr_images.npy'qr_labels_path = f'{QR_DATASET_PATH}/qr_labels.npy'# Option 1: Try to load from Kaggle dataset folders (if you've downloaded it)# Download from: https://www.kaggle.com/datasets/samahsadiq/benign-and-malicious-qr-codes# Expected structure:#   QR_DATASET_PATH/#     benign/#       *.png#     malicious/#       *.pngkaggle_qr_path = QR_DATASET_PATHqr_preprocessor_temp = QRCodePreprocessor(image_size=(128, 128))benign_folder = os.path.join(kaggle_qr_path, 'benign')malicious_folder = os.path.join(kaggle_qr_path, 'malicious')if os.path.exists(benign_folder) or os.path.exists(malicious_folder):    print("Loading QR code images from Kaggle dataset folders...")    print(f"  Benign folder: {benign_folder}")    print(f"  Malicious folder: {malicious_folder}")        qr_images, qr_labels = qr_preprocessor_temp.load_images_from_directory(        kaggle_qr_path    )        # Save as numpy arrays for faster loading next time    np.save(qr_images_path, qr_images)    np.save(qr_labels_path, qr_labels)    print(f"✓ Loaded {len(qr_images)} QR code images from Kaggle dataset")    print(f"✓ Saved to {qr_images_path} for faster loading next time")    # Option 2: Load from previously saved numpy arrayselif os.path.exists(qr_images_path) and os.path.exists(qr_labels_path):    print("Loading existing QR code dataset from Google Drive...")    qr_images = np.load(qr_images_path)    qr_labels = np.load(qr_labels_path)    print(f"✓ Loaded {len(qr_images)} QR code images from Drive")    # Option 3: Generate synthetic data as fallbackelse:    print("No existing QR dataset found. Generating synthetic QR code images...")    print("Tip: Download the Kaggle dataset and place folders at:")    print(f"  {benign_folder}/")    print(f"  {malicious_folder}/")        # Generate synthetic QR code images    qr_images, qr_labels = qr_preprocessor_temp.generate_synthetic_qr_images(        n_samples=5000,        malicious_ratio=0.3  # 30% malicious QR codes    )        # Save to Google Drive for future use    np.save(qr_images_path, qr_images)    np.save(qr_labels_path, qr_labels)    print(f"✓ Generated and saved {len(qr_images)} QR code images to Drive")# Display dataset infoprint("\n" + "="*60)print("QR CODE IMAGES DATASET (Image Data)")print("="*60)print(f"Images shape: {qr_images.shape}")print(f"Labels shape: {qr_labels.shape}")print(f"Malicious QR ratio: {qr_labels.mean():.2%}")print(f"Image value range: [{qr_images.min():.2f}, {qr_images.max():.2f}]")# Visualize sample QR codesfig, axes = plt.subplots(2, 4, figsize=(12, 6))fig.suptitle('Sample QR Code Images', fontsize=14, fontweight='bold')# Show benign QR codesbenign_indices = np.where(qr_labels == 0)[0][:4]for i, idx in enumerate(benign_indices):    axes[0, i].imshow(qr_images[idx])    axes[0, i].set_title('Benign', color='green')    axes[0, i].axis('off')# Show malicious QR codesmalicious_indices = np.where(qr_labels == 1)[0][:4]for i, idx in enumerate(malicious_indices):    axes[1, i].imshow(qr_images[idx])    axes[1, i].set_title('Malicious', color='red')    axes[1, i].axis('off')plt.tight_layout()plt.show()

## 6. FraudDataPreprocessor - Process Tabular Data

Apply preprocessing to transaction data using FraudDataPreprocessor.

In [ ]:
print("="*60)
print("STEP 1: FraudDataPreprocessor")
print("="*60)

# Initialize FraudDataPreprocessor
fraud_preprocessor = FraudDataPreprocessor()

# Preprocess tabular data
# This includes: encoding categorical features, scaling numeric features
X_tabular, y_tabular = fraud_preprocessor.preprocess_data(fraud_data, fit=True)

print(f"\n✓ Tabular data preprocessed")
print(f"  Input shape: {X_tabular.shape}")
print(f"  Labels shape: {y_tabular.shape}")
print(f"  Features: {fraud_preprocessor.feature_names}")
print(f"  Fraud ratio: {y_tabular.mean():.2%}")

# Show preprocessing details
print(f"\nPreprocessing applied:")
print(f"  1. Categorical encoding (transaction type)")
print(f"  2. Feature scaling (StandardScaler)")
print(f"  3. Reshape for transformer input: {X_tabular.shape}")

## 7. QRCodePreprocessor - Process Image Data

Apply preprocessing to QR code images using QRCodePreprocessor.

In [ ]:
print("="*60)
print("STEP 2: QRCodePreprocessor")
print("="*60)

# Initialize QRCodePreprocessor
qr_preprocessor = QRCodePreprocessor(image_size=(128, 128))

# Images are already preprocessed (normalized to [0, 1]) from generation
# In a real scenario, you would load raw images and preprocess them here
X_images = qr_images
y_qr = qr_labels

print(f"\n✓ Image data preprocessed")
print(f"  Input shape: {X_images.shape}")
print(f"  Labels shape: {y_qr.shape}")
print(f"  Image size: {X_images.shape[1:3]}")
print(f"  Value range: [{X_images.min():.2f}, {X_images.max():.2f}]")

print(f"\nPreprocessing applied:")
print(f"  1. Resize to (128, 128)")
print(f"  2. Normalize to [0, 1]")
print(f"  3. RGB format with 3 channels")

## 8. MultimodalDataPreprocessor - Combine Both Modalities

Use MultimodalDataPreprocessor to combine tabular and image data with correlated labels.

In [ ]:
print("="*60)
print("STEP 3: MultimodalDataPreprocessor")
print("="*60)

# Initialize MultimodalDataPreprocessor
multimodal_preprocessor = MultimodalDataPreprocessor(image_size=(128, 128))

# Prepare correlated multimodal data with train/test split
data = multimodal_preprocessor.prepare_train_test_data(
    test_size=0.2,
    random_state=42,
    n_samples=5000
)

print(f"\n✓ Multimodal data prepared")
print(f"\nTraining set:")
print(f"  Tabular: {data['X_train_tabular'].shape}")
print(f"  Images: {data['X_train_images'].shape}")
print(f"  Labels: {data['y_train'].shape}")
print(f"  Fraud ratio: {data['y_train'].mean():.2%}")

print(f"\nTest set:")
print(f"  Tabular: {data['X_test_tabular'].shape}")
print(f"  Images: {data['X_test_images'].shape}")
print(f"  Labels: {data['y_test'].shape}")
print(f"  Fraud ratio: {data['y_test'].mean():.2%}")

print(f"\nKey features:")
print(f"  1. Correlated modalities (fraud transactions → malicious QR codes)")
print(f"  2. Stratified train/test split")
print(f"  3. Both modalities preprocessed and aligned")

## 9. Save Processed Data to Google Drive

Store all preprocessed data to Google Drive for future use.

In [ ]:
print("="*60)
print("SAVING PROCESSED DATA TO GOOGLE DRIVE")
print("="*60)

# Save training data
np.save(f'{PROCESSED_DATA_PATH}/X_train_tabular.npy', data['X_train_tabular'])
np.save(f'{PROCESSED_DATA_PATH}/X_train_images.npy', data['X_train_images'])
np.save(f'{PROCESSED_DATA_PATH}/y_train.npy', data['y_train'])

# Save test data
np.save(f'{PROCESSED_DATA_PATH}/X_test_tabular.npy', data['X_test_tabular'])
np.save(f'{PROCESSED_DATA_PATH}/X_test_images.npy', data['X_test_images'])
np.save(f'{PROCESSED_DATA_PATH}/y_test.npy', data['y_test'])

# Save preprocessor states
multimodal_preprocessor.save_preprocessors(f'{PROCESSED_DATA_PATH}/preprocessors.pkl')

# Save metadata
metadata = {
    'n_train_samples': len(data['y_train']),
    'n_test_samples': len(data['y_test']),
    'train_fraud_ratio': float(data['y_train'].mean()),
    'test_fraud_ratio': float(data['y_test'].mean()),
    'tabular_shape': list(data['X_train_tabular'].shape),
    'image_shape': list(data['X_train_images'].shape),
    'image_size': [128, 128]
}

with open(f'{PROCESSED_DATA_PATH}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ All processed data saved to Google Drive")
print(f"  Location: {PROCESSED_DATA_PATH}")
print(f"\nSaved files:")
print(f"  - X_train_tabular.npy")
print(f"  - X_train_images.npy")
print(f"  - y_train.npy")
print(f"  - X_test_tabular.npy")
print(f"  - X_test_images.npy")
print(f"  - y_test.npy")
print(f"  - preprocessors.pkl")
print(f"  - metadata.json")

## 10. TabularTransformerEncoder - Process Transaction Features

Demonstrate the TabularTransformerEncoder processing transaction features.

In [ ]:
print("="*60)
print("TABULAR TRANSFORMER ENCODER")
print("="*60)

# Build TabularTransformerEncoder
model_builder = MultimodalFraudDetectionTransformer(MODEL_CONFIG)

# Create inputs
tabular_input = keras.Input(shape=(1, 8), name='tabular_input')

# Build tabular encoder
tabular_encoded = model_builder.build_tabular_encoder(tabular_input)

# Create model for demonstration
tabular_encoder_model = keras.Model(
    inputs=tabular_input,
    outputs=tabular_encoded,
    name='tabular_encoder'
)

print(f"\n✓ Tabular Transformer Encoder built")
print(f"\nArchitecture:")
print(f"  Input: {tabular_input.shape}")
print(f"  Dense projection → d_model={MODEL_CONFIG['d_model']}")
print(f"  + Positional encoding")
print(f"  TransformerBlock × {MODEL_CONFIG['num_layers']}")
print(f"    - Multi-head self-attention (heads={MODEL_CONFIG['num_heads']})")
print(f"    - Feed-forward network (dff={MODEL_CONFIG['dff']})")
print(f"  Output: {tabular_encoded.shape}")

# Process sample batch
sample_tabular = data['X_train_tabular'][:5]
encoded_output = tabular_encoder_model.predict(sample_tabular, verbose=0)

print(f"\n✓ Sample encoding:")
print(f"  Input shape: {sample_tabular.shape}")
print(f"  Output shape: {encoded_output.shape}")
print(f"  Output range: [{encoded_output.min():.3f}, {encoded_output.max():.3f}]")

# Visualize encoded features
plt.figure(figsize=(10, 4))
plt.imshow(encoded_output[0].T, aspect='auto', cmap='viridis')
plt.colorbar(label='Activation')
plt.title('Tabular Transformer Encoder Output (Sample 1)')
plt.xlabel('Sequence Position')
plt.ylabel('Feature Dimension')
plt.tight_layout()
plt.show()

## 11. VisionTransformerEncoder - Process QR Code Images

Demonstrate the VisionTransformerEncoder processing QR code images.

In [ ]:
print("="*60)
print("VISION TRANSFORMER ENCODER")
print("="*60)

# Create image input
image_input = keras.Input(shape=(128, 128, 3), name='image_input')

# Build vision encoder
image_encoded = model_builder.build_image_encoder(image_input)

# Create model for demonstration
vision_encoder_model = keras.Model(
    inputs=image_input,
    outputs=image_encoded,
    name='vision_encoder'
)

print(f"\n✓ Vision Transformer Encoder built")
print(f"\nArchitecture:")
print(f"  Input: {image_input.shape}")
print(f"  PatchEmbedding:")
print(f"    - Patch size: {MODEL_CONFIG.get('patch_size', 16)}×{MODEL_CONFIG.get('patch_size', 16)}")
print(f"    - Number of patches: {(128//MODEL_CONFIG.get('patch_size', 16))**2}")
print(f"    - Projection → d_model={MODEL_CONFIG['d_model']}")
print(f"  + Positional encoding")
print(f"  TransformerBlock × {MODEL_CONFIG['num_layers']}")
print(f"    - Multi-head self-attention (heads={MODEL_CONFIG['num_heads']})")
print(f"    - Feed-forward network (dff={MODEL_CONFIG['dff']})")
print(f"  Output: {image_encoded.shape}")

# Process sample batch
sample_images = data['X_train_images'][:5]
encoded_output = vision_encoder_model.predict(sample_images, verbose=0)

print(f"\n✓ Sample encoding:")
print(f"  Input shape: {sample_images.shape}")
print(f"  Output shape: {encoded_output.shape}")
print(f"  Number of patches: {encoded_output.shape[1]}")
print(f"  Output range: [{encoded_output.min():.3f}, {encoded_output.max():.3f}]")

# Visualize patches and encoded features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original image
axes[0].imshow(sample_images[0])
axes[0].set_title('Original QR Code Image')
axes[0].axis('off')

# Patch grid overlay
patch_size = MODEL_CONFIG.get('patch_size', 16)
img_with_grid = sample_images[0].copy()
axes[1].imshow(img_with_grid)
for i in range(0, 128, patch_size):
    axes[1].axhline(y=i, color='red', linewidth=0.5, alpha=0.5)
    axes[1].axvline(x=i, color='red', linewidth=0.5, alpha=0.5)
axes[1].set_title(f'Patch Grid ({patch_size}×{patch_size})')
axes[1].axis('off')

# Encoded features
axes[2].imshow(encoded_output[0].T, aspect='auto', cmap='viridis')
axes[2].set_title('Vision Transformer Encoder Output')
axes[2].set_xlabel('Patch Position')
axes[2].set_ylabel('Feature Dimension')

plt.colorbar(axes[2].images[0], ax=axes[2], label='Activation')
plt.tight_layout()
plt.show()

## 12. Complete Multimodal Model

Build and demonstrate the complete multimodal transformer model with cross-modal fusion.

In [ ]:
print("="*60)
print("COMPLETE MULTIMODAL TRANSFORMER MODEL")
print("="*60)

# Build complete multimodal model
model = model_builder.build_model()

print(f"\n✓ Complete multimodal model built")
print(f"\nArchitecture Overview:")
print(f"  1. Tabular Encoder: Transaction features → Embeddings")
print(f"  2. Vision Encoder: QR images → Patch embeddings")
print(f"  3. Cross-Modal Fusion: Bidirectional attention between modalities")
print(f"  4. Classification Head: Dense layers → Fraud probability")

model.summary()

# Compile model
model = model_builder.compile_model(learning_rate=TRAINING_CONFIG['learning_rate'])

print(f"\n✓ Model compiled")
print(f"  Optimizer: Adam (lr={TRAINING_CONFIG['learning_rate']})")
print(f"  Loss: Binary Crossentropy")
print(f"  Metrics: Accuracy, Precision, Recall, AUC")

## 13. Model Inference Demo

Demonstrate inference on sample data (without training).

In [ ]:
print("="*60)
print("INFERENCE DEMO (Untrained Model)")
print("="*60)

# Select sample data
sample_size = 10
sample_tabular = data['X_test_tabular'][:sample_size]
sample_images = data['X_test_images'][:sample_size]
sample_labels = data['y_test'][:sample_size]

# Make predictions
predictions = model.predict([sample_tabular, sample_images], verbose=0)

print(f"\n✓ Predictions made on {sample_size} samples")
print(f"\nResults (Note: Model is untrained, predictions are random):")
print(f"\n{'Sample':<8} {'True Label':<12} {'Predicted':<12} {'Probability':<12}")
print("-" * 50)

for i in range(sample_size):
    true_label = 'Fraud' if sample_labels[i] == 1 else 'Normal'
    pred_label = 'Fraud' if predictions[i][0] > 0.5 else 'Normal'
    prob = predictions[i][0]
    print(f"{i+1:<8} {true_label:<12} {pred_label:<12} {prob:<12.4f}")

# Visualize predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Predictions (Untrained Model)', fontsize=14, fontweight='bold')

for i in range(sample_size):
    row = i // 5
    col = i % 5
    
    axes[row, col].imshow(sample_images[i])
    
    true_label = 'Fraud' if sample_labels[i] == 1 else 'Normal'
    pred_label = 'Fraud' if predictions[i][0] > 0.5 else 'Normal'
    
    title = f'True: {true_label}\nPred: {pred_label} ({predictions[i][0]:.2f})'
    color = 'green' if true_label == pred_label else 'red'
    axes[row, col].set_title(title, fontsize=8, color=color)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print(f"\nNote: This model is untrained, so predictions are essentially random.")
print(f"To get meaningful results, the model needs to be trained on the data.")

## 14. Save Model to Google Drive

Save the model architecture to Google Drive.

In [ ]:
# Save model (architecture only, as it's untrained)
model_path = f'{MODEL_SAVE_PATH}/multimodal_model.keras'
model.save(model_path)

print("="*60)
print("MODEL SAVED TO GOOGLE DRIVE")
print("="*60)
print(f"\n✓ Model saved to: {model_path}")
print(f"\nTo train this model, use:")
print(f"  python src/train.py --mode multimodal")
print(f"\nOr in Colab:")
print(f"  !cd /content/test && python src/train.py --mode multimodal")

## 15. Pipeline Summary

Complete overview of what we've accomplished.

In [ ]:
print("="*70)
print("COMPLETE PIPELINE SUMMARY")
print("="*70)

summary = """
✓ STEP 1: Google Drive Setup
  - Mounted Google Drive
  - Created directory structure

✓ STEP 2: Data Loading
  - Loaded/Generated Online Payment Fraud Dataset (Tabular Data)
  - Loaded/Generated QR Code Images Dataset (Image Data)
  - All data stored in Google Drive

✓ STEP 3: Data Preprocessing
  - FraudDataPreprocessor: Processed transaction features
    * Encoded categorical variables
    * Scaled numeric features
    * Reshaped for transformer input
  
  - QRCodePreprocessor: Processed QR code images
    * Resized to 128×128
    * Normalized to [0, 1]
    * RGB format

✓ STEP 4: Multimodal Integration
  - MultimodalDataPreprocessor: Combined both modalities
    * Created correlated multimodal dataset
    * Stratified train/test split
    * Aligned tabular and image data

✓ STEP 5: Processed Data Storage
  - Saved all preprocessed data to Google Drive
  - Saved preprocessor states
  - Saved metadata

✓ STEP 6: Model Encoders
  - TabularTransformerEncoder: Processes transaction features
    * Self-attention mechanism
    * Positional encoding
    * Multi-layer transformer
  
  - VisionTransformerEncoder: Processes QR code images
    * Patch embedding (16×16 patches)
    * Positional encoding
    * Multi-layer transformer

✓ STEP 7: Complete Multimodal Model
  - Built complete model with cross-modal fusion
  - Bidirectional attention between modalities
  - Classification head for fraud detection

✓ STEP 8: Model Storage
  - Saved model to Google Drive

ALL DATA AND MODELS STORED IN GOOGLE DRIVE:
  {}

NEXT STEPS:
  1. Train the model using src/train.py
  2. Evaluate on test data
  3. Deploy for inference
""".format(DRIVE_BASE)

print(summary)

# Display storage statistics
print("\nStorage Statistics:")
print("-" * 70)

def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)  # Convert to MB

if os.path.exists(DRIVE_BASE):
    print(f"Datasets: {get_dir_size(FRAUD_DATASET_PATH) + get_dir_size(QR_DATASET_PATH):.2f} MB")
    print(f"Processed Data: {get_dir_size(PROCESSED_DATA_PATH):.2f} MB")
    print(f"Models: {get_dir_size(MODEL_SAVE_PATH):.2f} MB")
    print(f"Total: {get_dir_size(DRIVE_BASE):.2f} MB")

print("\n" + "="*70)
print("PIPELINE COMPLETE!")
print("="*70)